![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)


# M3L2 E15 - Model, Wrapper y Agent: la API moderna de LangChain (Resolution)

## Qué es este notebook

Este notebook **no reemplaza** a E00 ni a E06 — los actualiza. LangChain (desde su versión 1) agrega dos formas de trabajar que evitan escribir el wrapper a mano:

- `init_chat_model()`: crea un modelo a partir de un string `"proveedor:modelo"`, sin importar la clase específica (`ChatOpenAI`, `ChatAnthropic`, etc.).
- `create_agent()`: crea un agente completo (modelo + tools + loop de ejecución) con una sola función, en lugar de `create_tool_calling_agent()` + `AgentExecutor` (lo que viste en E06).

La pregunta que responde este notebook es: **si ya no escribo `ChatOpenAI` explícitamente, ¿el wrapper desapareció?** Spoiler: no. Solo se ocultó.


## BLOQUE 1 — Cinco piezas que no hay que confundir

### Las cinco piezas

| Elemento | Definición | Responsabilidad | Ejemplo |
|---|---|---|---|
| **Modelo real** | La IA entrenada o servida por un proveedor | Generar, analizar y decidir | GPT, Claude, Gemini, Llama |
| **Wrapper** | Adaptador entre LangChain y la API del proveedor | Traducir mensajes ↔ formato del proveedor | `ChatOpenAI` |
| **Model de LangChain** | Objeto con interfaz común (`BaseChatModel`) | Exponer `.invoke()`, `.stream()`, `.bind_tools()` | resultado de `init_chat_model()` |
| **Tool** | Función utilizable por el sistema | Consultar o ejecutar una acción puntual | `multiplicar()` |
| **Agent** | Orquestador con ciclo dinámico | Coordinar modelo, tools y estado hasta responder | `create_agent()` |

Error frecuente: pensar que `ChatOpenAI` **es** GPT. No lo es — es la clase que sabe *hablar* con la API de OpenAI. El modelo real vive del otro lado de esa llamada de red.


### Setup

Necesitamos la API key de OpenAI (igual que en el resto del módulo).

In [ ]:
import os, getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")


### Wrapper explícito (lo que ya conocías de E00)

```python
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
```


In [ ]:
from langchain_openai import ChatOpenAI

model_wrapper_explicito = ChatOpenAI(model="gpt-4o-mini", temperature=0)
respuesta = model_wrapper_explicito.invoke("Explica que es un wrapper en una oracion.")
print(f"Tipo: {type(model_wrapper_explicito).__name__}")
print(respuesta.content)


### La forma moderna: `init_chat_model()`

```python
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-4o-mini", temperature=0)
```

`init_chat_model` recibe un string `"proveedor:modelo"`, identifica qué integración usar (`langchain-openai`, `langchain-anthropic`, `langchain-ollama`, etc.) y la carga por vos. **Ese paquete de integración igual tiene que estar instalado** — `init_chat_model` no lo reemplaza, lo resuelve automáticamente.

```text
init_chat_model("openai:gpt-4o-mini")
                ↓
LangChain identifica el proveedor ("openai")
                ↓
carga la integracion correspondiente (ChatOpenAI)
                ↓
devuelve un BaseChatModel listo para usar
```


In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-4o-mini", temperature=0)
respuesta = model.invoke("Explica que es init_chat_model en una oracion.")
print(f"Tipo: {type(model).__name__}")
print(respuesta.content)
print()
print("Mismo tipo de objeto que ChatOpenAI: ambos son BaseChatModel con .invoke(), .stream(), .bind_tools()")


### ¿Cuándo usar cada forma?

| Situación | Forma recomendada |
|---|---|
| Introducción a LangChain / clase | `init_chat_model()` |
| Cambio frecuente de proveedor | `init_chat_model()` |
| Parámetros específicos de una integración (ej. `base_url` para un servidor local) | Wrapper explícito |
| Depuración de la integración | Wrapper explícito |
| Código corto para un agente | String `"proveedor:modelo"` directo en `create_agent()` |

**Relacionado con**: E00 - LLM Wrapper.


## BLOQUE 2 — Mensajes con roles

Un chat model recibe una lista de mensajes con rol, igual que en E00/E01. Los roles no cambiaron con la API moderna:

| Rol | Función |
|---|---|
| `system` | Define comportamiento e instrucciones globales |
| `user` | Mensaje del usuario |
| `assistant` | Respuestas previas del modelo |
| `tool` | Resultado de una tool, devuelto al modelo |


In [ ]:
respuesta = model.invoke([
    {"role": "system", "content": "Sos un instructor tecnico. Respondes con definiciones y un ejemplo."},
    {"role": "user", "content": "Cual es la diferencia entre un modelo y un wrapper?"},
])
print(respuesta.content)


## BLOQUE 3 — Tool calling no es lo mismo que Agent

Un modelo puede recibir tools sin convertirse automáticamente en un agente completo. `bind_tools()` solo le informa al modelo qué tools existen — **no ejecuta nada**.


In [ ]:
from langchain_core.tools import tool

@tool
def multiplicar(a: int, b: int) -> int:
    """Multiplica dos numeros enteros."""
    return a * b

model_con_tools = model.bind_tools([multiplicar])

respuesta = model_con_tools.invoke("Cuanto es 1537 multiplicado por 829?")
print(f"Contenido de texto: {respuesta.content!r}")
print(f"Tool calls solicitados: {respuesta.tool_calls}")
print()
print("El modelo todavia NO ejecuto multiplicar(). Solo generó la solicitud estructurada.")


Para completar el ciclo a mano habría que: leer el `tool_call`, ejecutar la función, agregar el resultado como `ToolMessage`, y volver a invocar al modelo. Ese ciclo completo es exactamente lo que hace un **Agent**.


## BLOQUE 4 — Crear un Agent con `create_agent()`

```text
Agent = Model + Tools + Instructions + State + Execution loop
```

En E06 armaste este mismo ciclo con `create_tool_calling_agent()` + `AgentExecutor(verbose=True, max_iterations=5)`. `create_agent()` es la evolución de alto nivel de ese mismo patrón (por dentro usa capacidades de LangGraph).


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[multiplicar],
    system_prompt="Sos un asistente matematico. Usa la herramienta disponible para realizar multiplicaciones.",
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Cuanto es 1537 multiplicado por 829?"}
    ]
})

print(result["messages"][-1].content)


### Qué ocurre internamente

```text
1. El usuario formula una pregunta.
2. El agente envia los mensajes al modelo.
3. El modelo detecta que existe una tool apropiada.
4. El modelo solicita multiplicar(a=1537, b=829).
5. El agente ejecuta la funcion.
6. La tool devuelve el resultado.
7. El agente agrega el resultado al estado (ToolMessage).
8. El modelo produce la respuesta final.
```


### Agent con varias tools

In [ ]:
@tool
def consultar_cliente(email: str) -> str:
    """Busca un cliente mediante su correo electronico."""
    clientes = {
        "ana@example.com": "Ana - plan Premium - cuenta activa",
        "juan@example.com": "Juan - plan Basico - pago pendiente",
    }
    return clientes.get(email, "Cliente no encontrado")

@tool
def calcular_descuento(precio: float, porcentaje: float) -> float:
    """Calcula el precio final aplicando un porcentaje de descuento."""
    return round(precio - (precio * porcentaje / 100), 2)

agent_comercial = create_agent(
    model="openai:gpt-4o-mini",
    tools=[consultar_cliente, calcular_descuento],
    system_prompt=(
        "Sos un asistente comercial. Usa herramientas cuando necesites consultar "
        "clientes o calcular precios. No inventes informacion que las herramientas no devuelvan."
    ),
)

result = agent_comercial.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Consulta a ana@example.com y calcula cuanto pagaria "
                "por un producto de 250 dolares con 15% de descuento."
            ),
        }
    ]
})

print(result["messages"][-1].content)


## BLOQUE 5 — Agent vs Chain/workflow

| Criterio | Chain / workflow | Agent |
|---|---|---|
| Flujo | Definido por código | Decidido parcialmente por el modelo |
| Previsibilidad | Alta | Menor |
| Costo computacional | Menor | Mayor (puede iterar varias veces) |
| Depuración | Más simple | Más compleja |
| Tools | Opcionales y explícitas | Elegidas dinámicamente por el modelo |
| Uso recomendado | Procesos con pasos conocidos | El modelo debe decidir qué hacer |

```text
Si conoces los pasos       -> chain o workflow (prompt | model | parser)
Si el modelo debe decidir  -> agent (create_agent)
```

**Relacionado con**: E03 - LCEL Chain, E06 - Agente completo.


## BLOQUE 6 — Bonus: el mismo agente con un modelo local (Ollama)

`create_agent()` acepta cualquier string `"proveedor:modelo"` que `init_chat_model` reconozca — incluido un modelo local. Esta celda es opcional: se saltea sola si no tenés Ollama corriendo.


In [ ]:
# Ollama (modelo local, sin costo ni API key)
# 1) Instalar Ollama: https://ollama.com
# 2) ollama pull llama3.2
# 3) pip install langchain-ollama

try:
    agent_local = create_agent(
        model="ollama:llama3.2",
        tools=[multiplicar],
        system_prompt="Usa la herramienta para resolver multiplicaciones.",
    )
    result_local = agent_local.invoke({
        "messages": [{"role": "user", "content": "Cuanto es 12 multiplicado por 8?"}]
    })
    print(result_local["messages"][-1].content)
except ImportError:
    print("Salteado: falta instalar langchain-ollama (pip install langchain-ollama).")
except Exception as e:
    print("Salteado: no se pudo conectar a Ollama local. Verifica que 'ollama serve' este corriendo.")
    print(f"Detalle: {e}")


> El modelo local elegido debe soportar bien tool calling para que el agente sea confiable — no todos los modelos pequeños lo hacen.


## Errores conceptuales frecuentes

| Error | Por qué está mal |
|---|---|
| "El agent reemplaza al model" | El agente **usa** un modelo; sin él no puede decidir nada |
| "Ya no existen wrappers" | Siguen existiendo, `init_chat_model` solo oculta su creación |
| "`ChatOpenAI` es GPT" | `ChatOpenAI` es la integración; GPT es el modelo real |
| "Toda app con tools es un agente" | `model.bind_tools(...)` sin loop de ejecución no es un agente |
| "Un agente siempre es mejor" | Un agente puede costar más, tardar más y ser menos predecible |
| "El modelo ejecuta las funciones Python" | El modelo **solicita**; el runtime del agente **ejecuta** |


## Resumen — Lo que demuestra E15

```text
Modelo real         = inteligencia artificial entrenada o servida
Wrapper             = adaptador entre LangChain y el proveedor
Model de LangChain  = objeto invocable con interfaz comun (init_chat_model)
Tool                = funcion con esquema definido (@tool)
Agent               = Model + Tools + Instructions + State + Loop (create_agent)
Workflow            = pasos definidos por el desarrollador (prompt | model | parser)
```

**El wrapper no desapareció — LangChain solo aprendió a crearlo por vos.**


## Checks automáticos

In [ ]:
def run_checks():
    from langchain_core.messages import AIMessage

    assert isinstance(model, ChatOpenAI) or type(model).__name__ == "ChatOpenAI"
    r = model.invoke("Di solo: test")
    assert isinstance(r, AIMessage)

    r_tools = model_con_tools.invoke("Cuanto es 3 multiplicado por 4?")
    assert len(r_tools.tool_calls) >= 1
    assert r_tools.tool_calls[0]["name"] == "multiplicar"

    result = agent.invoke({"messages": [{"role": "user", "content": "Cuanto es 6 multiplicado por 7?"}]})
    assert "42" in result["messages"][-1].content

    print("M3L2 E15 Resolution checks passed")


run_checks()


## Referencias oficiales

- [LangChain — Models](https://docs.langchain.com/oss/python/langchain/models)
- [LangChain — Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain — Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [LangChain Reference — init_chat_model](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model)
- [LangChain Reference — create_agent](https://reference.langchain.com/python/langchain/agents/factory/create_agent)
